#Analise exploratória de dados

In [0]:
#cliente
#produtos - categorias

# Quantos pedidos cada cliente fez?
# Quantas categorias diferentes comprou?
# Quanto gastou em cada categoria?
# Qual sua categoria favorita?
# Qual categoria representa 80% dos gastos dele?

In [0]:
# olist_orders_dataset(id_unico, id) -customer_id - olist_customers_dataset (id_unico, id, customer_zip_code_prefix)

# id_cliente
# id_categoria (y)

# #analise exploratoroia de dados
# #correlacionar features
# #quais melhores features?
# #tecnicas de analise exploratoria

##Data Definitions

In [0]:
# [file.name for file in dbutils.fs.ls(f'{volume}') if file.name.endswith('.csv')]

# ['olist_customers_dataset.csv',
#  'olist_geolocation_dataset.csv',
#  'olist_order_items_dataset.csv',
#  'olist_order_payments_dataset.csv',
#  'olist_order_reviews_dataset.csv',
#  'olist_orders_dataset.csv',
#  'olist_products_dataset.csv',
#  'olist_sellers_dataset.csv',
#'product_category_name_translation.csv']

In [0]:
volume = '/Workspace/Users/greyce.costa@thoughtworks.com/ml_training_dev.olist/raw/files'
catalog = "ml_training_dev"
schema = "silver"

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS {}".format(catalog))
spark.sql("CREATE SCHEMA IF NOT EXISTS {}".format(schema))

##Imports

In [0]:
from pyspark.sql.functions import current_date, count, col, sum, countDistinct, row_number
from pyspark.sql.window import Window

##olist_order_reviews_dataset

In [0]:
file_name = "olist_order_reviews_dataset.csv"
df_reviews = spark.read.csv(f"{volume}/{file_name}", header=True, inferSchema=True)
df_reviews.display()

In [0]:
df_reviews.where(col("order_id")== "ea147e73648571e7ae04cfc2ba96aa35").display()

In [0]:
# total = df.count()
# nao_null = df.select("review_comment_message").distinct().count()

##olist_products_dataset

In [0]:
file_name = "olist_products_dataset.csv"
df_products = spark.read.csv(f"{volume}/{file_name}", header=True, inferSchema=True)
df_products.limit(10).display()

##olist_order_items_dataset

In [0]:
file_name = "olist_order_items_dataset.csv"
df_order_items = spark.read.csv(f"{volume}/{file_name}", header=True, inferSchema=True)
# df_order_items.filter(col('order_id') == "00143d0f86d6fbd9f9b38ab440ac16f5").display()

In [0]:
# dupl = df_order_items.groupBy("order_id", "order_item_id").count()
# dupl.orderBy(col("count").desc()).limit(10).display()

##olist_orders_dataset

In [0]:
file_name = "olist_orders_dataset.csv"
df_olist_orders_dataset = spark.read.csv(f"{volume}/{file_name}", header=True, inferSchema=True)
df_olist_orders_dataset.limit(10).display()

##olist_customers_dataset

In [0]:
file_name = "olist_customers_dataset.csv"
df_customers = spark.read.csv(f"{volume}/{file_name}", header=True, inferSchema=True)
df_customers.limit(10).display()

##product_category_name_translation

In [0]:

# file_name = "product_category_name_translation.csv"
# df_products = spark.read.csv(f"{volume}/{file_name}", header=True, inferSchema=True)
# df_products.limit(10).display()

##olist_sellers_dataset

In [0]:
# file_name = "olist_sellers_dataset.csv"
# df_products = spark.read.csv(f"{volume}/{file_name}", header=True, inferSchema=True)
# df_products.limit(10).display()

##olist_geolocation_dataset

In [0]:
# file_name = "olist_geolocation_dataset.csv"
# raw_df = spark.read.csv(f"{volume}/{file_name}", header=True, inferSchema=True)
# raw_df.dropDuplicates()
# raw_tranformed = raw_df.withColumn("loadDate", current_date())
# display(raw_tranformed)

In [0]:
# dim_location = spark.table("ml_training_dev.gold.dim_location").select(col("customer_zip_code_prefix").alias('geolocation_zip_code_prefix')).distinct()

In [0]:
# zip_not_in_olist_geolocation =dim_location.join(raw_tranformed, on='geolocation_zip_code_prefix', how='left')
# zip_not_in_olist_geolocation.where(col('loadDate').isNull()).display()

In [0]:
# raw_tranformed.where(col('customer_unique_id') == '8d50f5eadf50201ccdcedfb9e2ac8455').display()
# raw_tranformed.where(col('customer_unique_id') == '3e43e6105506432c953e165fb2acf44c').display()


## Perguntas

###Quality

In [0]:
def records_df(df1, df2, result) -> bool:
    df1_size= df1.count()
    df2_size= df2.count()
    result_size= result.count()

    bigger_df = df2_size if df1_size < df2_size else df1_size

    if result_size > bigger_df:
        return  True
    else:
        return False

###Quantos pedidos cada cliente fez? (group by customer_unique_id, order_id)

In [0]:
df_olist_orders_dataset
# df_olist_orders_dataset.where(col('customer_id').isNull()).count()

In [0]:
df_customers
# df_customers.where(col('customer_id').isNull()).count()

In [0]:
df_customers_orders = df_customers.join(df_olist_orders_dataset,on = ["customer_id"],how = "left")

valida_explosao = records_df(df_customers, df_olist_orders_dataset, df_customers_orders)
if valida_explosao:
    print("ERRO: explosão de dados")
else:
    print("OK: explosão de dados")

####Resposta

In [0]:
df_customers_orders.groupBy("customer_unique_id").agg(countDistinct("order_id").alias("num_orders")).orderBy(col("num_orders").desc()).display()

###Quantas categorias diferentes comprou? (group by product_category_name, count(order_item_id))

In [0]:
df_products

In [0]:
# df_order_items.groupBy("order_id", "order_item_id","product_id").count().orderBy(col("count").desc()).limit(5).show()
df_order_items

In [0]:
# df_customers_orders.groupBy("order_id").count().orderBy(col("count").desc()).limit(5).show()

##df_customers_orders_category

In [0]:
df_customers_orders_category = (df_customers_orders
                                .join(df_order_items, on = ["order_id"], how = "left")
                                .join(df_products, on = ["product_id"], how = "left")
                                )
valida_explosao = records_df(df_products, df_order_items, df_customers_orders_category)
if valida_explosao:
    print("ERRO: explosão de dados")
else:
    print("OK: explosão de dados")

####Resposta

In [0]:
#Quantas categorias diferentes comprou?
df_categoria_distinct = df_customers_orders_category.groupBy("customer_unique_id").agg(countDistinct("product_category_name").alias("product_category_qtt")).orderBy(col("product_category_qtt").desc()).display()

###Quanto gastou em cada categoria?

####Resposta

In [0]:
# Quanto gastou em cada categoria?
df_gasto_categoria = df_customers_orders_category.groupBy("customer_unique_id","product_category_name").agg(sum("price").alias("gasto")).display()

###Qual sua categoria favorita?

In [0]:
#Quantas categorias diferentes comprou?
df_categoria_favorita = (df_customers_orders_category
                         .groupBy("customer_unique_id", "product_category_name")
                         .agg(countDistinct("order_id").alias("orders_qtt"),
                              sum("price").alias("gasto"))
                         .withColumn("row_number", row_number().over(Window.partitionBy("customer_unique_id").orderBy(col("orders_qtt").desc(), col("gasto").desc())))
                         )#.where("row_number = 1").drop("row_number", "orders_qtt", "gasto")

In [0]:
df_categoria_favorita.filter(col("customer_unique_id") == "8f6ce2295bdbec03cd50e34b4bd7ba0a").display()

In [0]:
df_categoria_favorita.display()

###Qual categoria representa 80% dos gastos dele?``

In [0]:
df_customers_orders_category = df_customers_orders_category.dropDuplicates().dropna()
df_customers_orders_category.display()

In [0]:
# df_reviews.groupBy("order_id").count().orderBy(col("count").desc()).display()

In [0]:
# df_customers_orders_category.where(col("order_id")== "ea147e73648571e7ae04cfc2ba96aa35").display()

In [0]:
# df_customers_orders_category.display()

In [0]:
# df_customers_orders_category = df_customers_orders_category.filter(~col("is_high_spender").isin([1, 2]))

###LogisticRegression

In [0]:
df_sample = df_customers_orders_category.sample(fraction=0.1, seed=42)
df_train, df_test = df_sample.randomSplit([0.8, 0.2], seed=42)
display(df_train)

In [0]:
display(df_test)

In [0]:
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import LogisticRegression

# Define binary target column based on 'price'
threshold = 1000
df_train = df_train.withColumn("is_high_spender", (col("price") > threshold).cast("integer"))
df_test = df_test.withColumn("is_high_spender", (col("price") > threshold).cast("integer"))

In [0]:
df_test.display()

In [0]:
# Index categorical columns
indexers = [
    StringIndexer(inputCol="product_category_name", outputCol="product_category_name_index"),
    StringIndexer(inputCol="customer_unique_id", outputCol="customer_unique_id_index")
]
for indexer in indexers:
    model = indexer.fit(df_train)
    df_train = model.transform(df_train)
    df_test = model.transform(df_test)
    del model

In [0]:
# Assemble features
feature_cols = ["product_category_name_index", "customer_unique_id_index", "price"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_train = assembler.transform(df_train)
df_test = assembler.transform(df_test)

# Fit logistic regression model
lr = LogisticRegression(featuresCol="features", labelCol="is_high_spender")
model = lr.fit(df_train)

# Display feature weights (coefficients)
display(model.coefficients)

In [0]:
# df_lr=df_lr.filter(col("is_high_spender).isIn(0,1)")
# df_lr.limit(5).display()

In [0]:
for indexer in indexers:
    model = indexer.fit(df_lr); df_lr = model.transform(df_lr); del model
# Após esse passo, df_lr contém novas colunas indexadas, mas display falha se você tentar exibir apenas model.coefficients, pois model.coefficients não é um DataFrame.

# Select features for logistic regression
feature_cols = ["product_category_name_index", "customer_unique_id_index", "price"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_lr = assembler.transform(df_lr)

In [0]:
# df_lr = df_lr.drop('product_category_name_index').drop('customer_unique_id_index')

In [0]:
# for indexer in indexers:
#     model = indexer.fit(df_lr)
#     df_lr = model.transform(df_lr)
#     del model
#     df_lr = df_lr.drop('product_category_name_index').drop('customer_unique_id_index')

In [0]:
# Fit logistic regression model
lr = LogisticRegression(featuresCol="features", labelCol="is_high_spender")
model = lr.fit(df_lr)

# Display feature weights (coefficients)
display(model.coefficients)

##Writting table

In [0]:
# target_table = "nome_aqui"

In [0]:
# raw_tranformed.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.{target_table}")
# print("table successfully created!")